# Microbiology Workflow Analysis

## Objective
This project analyzes synthetic microbiology and molecular workflow data modeled after Epic Beaker-aligned laboratory operations.

## Investigation Focus
- Where are turnaround time delays occurring?
- Which locations show the greatest transport bottlenecks?
- Are STAT workflows meeting expected performance?
- Are delays primarily pre-analytic, transport-related, or lab-related?

In [3]:
from microbiology_workflow_analysis_generator import (
    build_orders,
    add_workflow_metrics,
    add_event_flags,
)

df = build_orders()
df = add_workflow_metrics(df)
df = add_event_flags(df)

df.head()

,order_id,patient_location,priority,test_name,department,instrument,specimen_type,workflow_type,order_time,collect_time,...,tat_transport_min,tat_lab_min,tat_result_post_min,tat_total_min,stat_delay_flag,tat_total_hours,collection_delay_flag,transport_delay_flag,lab_delay_flag,result_post_delay_flag
0,1,ICU,STAT,Wound Culture,Microbiology,Manual Culture,Wound,Culture,2026-04-11 03:26:29.601689,2026-04-11 03:38:29.601689,...,9.0,1859.0,9.0,1889.0,Y,31.483333,N,N,Y,N
1,2,ICU,STAT,COVID/Flu PCR,Molecular,ID NOW,Nasal Swab,Molecular,2026-04-11 18:33:29.601919,2026-04-11 18:57:29.601919,...,5.0,101.0,9.0,139.0,N,2.316667,N,N,N,N
2,3,MedSurg,Routine,Urine Culture,Microbiology,Manual Culture,Urine,Culture,2026-04-19 06:21:29.602057,2026-04-19 07:30:29.602057,...,63.0,1649.0,6.0,1787.0,N,29.783333,Y,N,Y,N
3,4,MedSurg,Routine,COVID/Flu PCR,Molecular,ID NOW,Nasal Swab,Molecular,2026-03-31 02:46:29.602181,2026-03-31 03:06:29.602181,...,78.0,197.0,7.0,302.0,N,5.033333,N,N,N,N
4,5,PrimaryCare,Routine,MRSA Screen,Microbiology,BD Max,Nasal Swab,Molecular,2026-03-22 23:27:29.602308,2026-03-23 02:24:29.602308,...,218.0,240.0,4.0,639.0,N,10.650000,Y,Y,N,N


## Dataset Overview
This dataset simulates microbiology and molecular test orders across multiple patient locations, priorities, and workflow types.

It includes timestamp-based workflow stages and derived metrics for:
- collection delay
- transport delay
- lab processing time
- result posting time
- total turnaround time

In [16]:
print("Rows, Columns:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nPatient Location Distribution:")
print(df["patient_location"].value_counts())

print("\nPriority Distribution:")
print(df["priority"].value_counts())

print("\nWorkflow Type Distribution:")
print(df["workflow_type"].value_counts())

Rows, Columns: (1500, 27)

Columns:
['order_id', 'patient_location', 'priority', 'test_name', 'department', 'instrument', 'specimen_type', 'workflow_type', 'order_time', 'collect_time', 'receive_time', 'instrument_result_time', 'final_result_time', 'downstream_post_status', 'verification_status', 'result_status', 'tat_collection_min', 'tat_transport_min', 'tat_lab_min', 'tat_result_post_min', 'tat_total_min', 'stat_delay_flag', 'tat_total_hours', 'collection_delay_flag', 'transport_delay_flag', 'lab_delay_flag', 'result_post_delay_flag']

Patient Location Distribution:
patient_location
PrimaryCare    277
ED             271
Oncology       258
FSED           236
MedSurg        234
ICU            224
Name: count, dtype: int64

Priority Distribution:
priority
Routine    1183
STAT        317
Name: count, dtype: int64

Workflow Type Distribution:
workflow_type
Molecular    882
Culture      618
Name: count, dtype: int64


## Overall Turnaround Time Profile
The first step is to evaluate total and stage-level turnaround time across the full dataset to identify which workflow segments contribute most to overall delay.

In [17]:
df[[
    "tat_collection_min",
    "tat_transport_min",
    "tat_lab_min",
    "tat_result_post_min",
    "tat_total_min",
    "tat_total_hours"
]].describe().round(2)

,tat_collection_min,tat_transport_min,tat_lab_min,tat_result_post_min,tat_total_min,tat_total_hours
count,1500.00,1500.00,1500.00,1500.00,1500.00,1500.00
mean,46.80,90.19,976.54,10.79,1124.31,18.74
std,37.15,106.16,1127.58,24.36,1141.86,19.03
min,5.00,5.00,30.00,1.00,64.00,1.07
25%,21.00,25.00,116.00,3.00,221.75,3.70
50%,34.00,43.00,208.00,6.00,422.00,7.03
75%,62.25,99.00,1882.25,9.00,1996.00,33.27
max,179.00,480.00,4300.00,239.00,4676.00,77.93


## Turnaround Time by Patient Location
Location is one of the most important workflow variables because inpatient areas, emergency departments, and outreach-style sites often behave very differently operationally.

In [18]:
df.groupby("patient_location")["tat_total_hours"].mean().sort_values(ascending=False).round(2)

patient_location
PrimaryCare    22.21
FSED           20.93
MedSurg        20.15
Oncology       17.13
ICU            16.80
ED             15.20
Name: tat_total_hours, dtype: float64

In [19]:
df.groupby("patient_location")[[
    "tat_collection_min",
    "tat_transport_min",
    "tat_lab_min",
    "tat_result_post_min"
]].mean().round(1)

,tat_collection_min,tat_transport_min,tat_lab_min,tat_result_post_min
patient_location,,,,
ED,26.2,21.8,854.6,9.6
FSED,46.0,133.5,1066.1,10.5
ICU,24.7,22.6,949.2,11.4
MedSurg,44.8,52.2,1102.0,9.8
Oncology,43.8,48.8,922.0,12.9
PrimaryCare,90.0,245.4,986.5,10.6


### Interpretation
This view helps identify whether high total turnaround times are primarily driven by:
- delayed collection
- delayed transport / receipt
- lab processing time
- downstream result posting

Outreach-like locations such as FSED and PrimaryCare are expected to show greater transport variability than ICU or ED workflows.

## STAT vs Routine Performance
Priority handling is critical in clinical operations. This section evaluates whether STAT workflows are consistently achieving faster turnaround time than routine testing.

In [21]:
df.groupby("priority")["tat_total_hours"].mean().round(2)

priority
Routine    19.71
STAT       15.10
Name: tat_total_hours, dtype: float64

In [22]:
df.groupby("priority")[[
    "tat_collection_min",
    "tat_transport_min",
    "tat_lab_min",
    "tat_result_post_min"
]].mean().round(1)

,tat_collection_min,tat_transport_min,tat_lab_min,tat_result_post_min
priority,,,,
Routine,54.5,104.6,1012.9,10.9
STAT,17.9,36.3,841.0,10.4


In [23]:
df["stat_delay_flag"].value_counts()

stat_delay_flag
N    1356
Y     144
Name: count, dtype: int64

## Bottleneck Flag Analysis
Threshold-based event flags simulate the type of operational alert logic often used in reporting layers such as Epic Clarity or workflow dashboards.

In [24]:
flag_cols = [
    "collection_delay_flag",
    "transport_delay_flag",
    "lab_delay_flag",
    "result_post_delay_flag",
    "stat_delay_flag",
]

for col in flag_cols:
    print(f"\n{col}")
    print(df[col].value_counts())


collection_delay_flag
collection_delay_flag
N    1110
Y     390
Name: count, dtype: int64

transport_delay_flag
transport_delay_flag
N    1151
Y     349
Name: count, dtype: int64

lab_delay_flag
lab_delay_flag
N    882
Y    618
Name: count, dtype: int64

result_post_delay_flag
result_post_delay_flag
N    1406
Y      94
Name: count, dtype: int64

stat_delay_flag
stat_delay_flag
N    1356
Y     144
Name: count, dtype: int64


In [25]:
df[df["transport_delay_flag"] == "Y"]["patient_location"].value_counts()

patient_location
PrimaryCare    222
FSED           127
Name: count, dtype: int64

In [26]:
df[df["stat_delay_flag"] == "Y"][[
    "order_id",
    "patient_location",
    "priority",
    "test_name",
    "tat_total_hours",
    "tat_transport_min",
    "tat_lab_min"
]].head(10)

,order_id,patient_location,priority,test_name,tat_total_hours,tat_transport_min,tat_lab_min
0,1,ICU,STAT,Wound Culture,31.483333,9.0,1859.0
11,12,PrimaryCare,STAT,Wound Culture,68.850000,32.0,4065.0
17,18,ED,STAT,Urine Culture,20.700000,12.0,1218.0
21,22,FSED,STAT,Blood Culture,29.033333,41.0,1688.0
24,25,FSED,STAT,COVID/Flu PCR,3.083333,38.0,116.0
25,26,FSED,STAT,Urine Culture,39.533333,104.0,2237.0
29,30,FSED,STAT,Wound Culture,39.366667,73.0,2273.0
32,33,ED,STAT,Blood Culture,29.583333,11.0,1742.0
33,34,PrimaryCare,STAT,Urine Culture,32.766667,26.0,1924.0
38,39,Oncology,STAT,Wound Culture,38.866667,57.0,2247.0


## Workflow Type Comparison
Molecular and culture workflows have inherently different processing characteristics. This section compares expected performance differences across workflow types and test categories.

In [27]:
df.groupby("workflow_type")["tat_total_hours"].mean().round(2)

workflow_type
Culture      38.83
Molecular     4.66
Name: tat_total_hours, dtype: float64

In [28]:
df.groupby("test_name")["tat_total_hours"].mean().sort_values(ascending=False).round(2)

test_name
Wound Culture        49.13
Urine Culture        35.72
Blood Culture        32.16
MRSA Screen           4.84
C. Diff PCR           4.74
Respiratory Panel     4.55
COVID/Flu PCR         4.50
Name: tat_total_hours, dtype: float64

# Key Findings

1. Transport delays are a major contributor to prolonged turnaround time, especially in outreach-style or off-site workflows.
2. STAT tests perform better overall than routine tests, but a subset still exceeds expected turnaround thresholds.
3. Workflow type strongly influences TAT, with culture-based workflows showing predictably longer processing times than molecular assays.
4. Delay segmentation helps distinguish whether performance issues are driven by collection, transport, lab processing, or downstream posting.

# Operational Recommendations

- Review specimen transport workflows for FSED and PrimaryCare locations
- Monitor delayed STAT orders using threshold-based alert logic
- Separate collection, transport, and lab timing in workflow reporting views
- Extend the workflow model into SQL-based Clarity-style investigation for deeper operational troubleshooting